# Agent Network: Data Cleaning & Summary
**Engagement 2:** audit, clean, merge and summarise the mobile-money agent transactions.
Cleaning is done by `clean_agent_data()` in `cleaning_pipeline.py`.

In [1]:
!git clone https://github.com/mangacindy9-code/plp-da-a3-duka-sales-and-agent-cleanup.git
%cd plp-da-a3-duka-sales-and-agent-cleanup
!ls -a

fatal: destination path 'plp-da-a3-duka-sales-and-agent-cleanup' already exists and is not an empty directory.
/content/plp-da-a3-duka-sales-and-agent-cleanup
.			.gitignore
..			outputs
agents.csv		plp-da-a3-duka-sales-and-agent-cleanup
agent_transactions.csv	__pycache__
cleaning_pipeline.py	README.md
generate_data.py	requirements.txt
.git


In [2]:
import os
import pandas as pd, numpy as np
from cleaning_pipeline import clean_agent_data

os.makedirs("outputs", exist_ok=True)

# Fix generate_data.py by removing the %%writefile magic line if present
if os.path.exists("generate_data.py"):
    with open("generate_data.py", "r") as f:
        lines = f.readlines()
    if lines and "%%writefile" in lines[0]:
        with open("generate_data.py", "w") as f:
            f.writelines(lines[1:])

# Generate the data files
!python generate_data.py

txns_raw = pd.read_csv("agent_transactions.csv")
agents_raw = pd.read_csv("agents.csv")
# Second read WITHOUT NA conversion, to separate empty cells from literal "N/A"
txns_str = pd.read_csv("agent_transactions.csv", dtype=str, keep_default_na=False)

print("Transactions:", txns_raw.shape, "| Agents:", agents_raw.shape)
display(txns_raw.head())
txns_raw.info()

Transactions: (612, 5) | Agents: (20, 3)


,txn_id,agent_id,txn_type,amount,txn_date
0,TX00001,AG009,deposit,353.0,2026-03-03
1,TX00002,AG020,deposit,1238.0,2026-04-27
2,TX00003,AG008,DEPOSIT,9129.0,2026-04-29
3,TX00004,AG009,deposit,652.0,2026-04-24
4,TX00005,AG009,withdrawal,478.0,2026-04-17


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 612 entries, 0 to 611
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   txn_id    612 non-null    object 
 1   agent_id  612 non-null    object 
 2   txn_type  612 non-null    object 
 3   amount    590 non-null    float64
 4   txn_date  612 non-null    object 
dtypes: float64(1), object(4)
memory usage: 24.0+ KB


## 1. Data-quality audit
Issues are counted on the **raw** data. pandas turns the text "N/A" into NaN by default,
so I re-read the file with `keep_default_na=False` to separate empty cells from literal "N/A".

In [3]:
amt_num = pd.to_numeric(txns_raw["amount"], errors="coerce")
q1, q3 = amt_num.quantile([.25, .75]); iqr = q3 - q1
iqr_out = (amt_num < q1 - 1.5*iqr) | (amt_num > q3 + 1.5*iqr)

amt_s = txns_str["amount"].str.strip()
dates_iso = pd.to_datetime(txns_str["txn_date"].str.replace("/", "-"), errors="coerce")

audit = pd.DataFrame([
    ("Repeated transaction records (same txn_id)", "Uniqueness", "txn_id",
        txns_raw.duplicated("txn_id").sum()),
    ("Inconsistent txn_type labels (case/whitespace)", "Consistency", "txn_type",
        (~txns_raw["txn_type"].isin(["Deposit", "Withdrawal"])).sum()),
    ("Dates in YYYY/MM/DD instead of YYYY-MM-DD", "Consistency", "txn_date",
        txns_str["txn_date"].str.contains("/").sum()),
    ("Amount empty (NaN)", "Completeness", "amount", (amt_s == "").sum()),
    ('Amount is the text "N/A"', "Completeness", "amount", (amt_s == "N/A").sum()),
    ("agent_id not in agent register", "Validity", "agent_id",
        (~txns_raw["agent_id"].isin(agents_raw["agent_id"])).sum()),
    ("Amount exactly 250,000 (far above normal range)", "Accuracy", "amount",
        (amt_num == 250000).sum()),
    ("Amount outside IQR fences (1.5×IQR)", "Accuracy", "amount", iqr_out.sum()),
    ("Dates unparseable or in the future (0 = check passed)", "Timeliness", "txn_date",
        (dates_iso.isna() | (dates_iso > pd.Timestamp.today())).sum()),
], columns=["issue", "dimension", "column", "rows_affected"])
audit["rows_affected"] = audit["rows_affected"].astype(int)
audit

,issue,dimension,column,rows_affected
0,Repeated transaction records (same txn_id),Uniqueness,txn_id,12
1,Inconsistent txn_type labels (case/whitespace),Consistency,txn_type,384
2,Dates in YYYY/MM/DD instead of YYYY-MM-DD,Consistency,txn_date,5
3,Amount empty (NaN),Completeness,amount,15
4,"Amount is the text ""N/A""",Completeness,amount,7
5,agent_id not in agent register,Validity,agent_id,8
6,"Amount exactly 250,000 (far above normal range)",Accuracy,amount,5
7,Amount outside IQR fences (1.5×IQR),Accuracy,amount,40
8,Dates unparseable or in the future (0 = check ...,Timeliness,txn_date,0


## 2. Run the cleaning pipeline
All rules live in `cleaning_pipeline.py` (no hard-coded rows). The log below documents each step.

In [4]:
clean, log = clean_agent_data(txns_raw, agents_raw)
display(log)
print(f"Row count before: {len(txns_raw)} | after: {len(clean)} | removed: {len(txns_raw)-len(clean)}")
clean.dtypes

,step,rule,rows_affected,decision
0,0 Input,Raw transaction rows,612,
1,1 Consistency,Strip whitespace in txn_id,0,
2,1 Consistency,Strip whitespace in agent_id,0,
3,1 Consistency,Strip whitespace in txn_type,111,
4,2 Uniqueness,Drop repeated txn_id (keep first),12,Repeated records of the same transaction are d...
5,3 Consistency,Standardise txn_type case,376,
6,3 Validity,txn_type not mappable (left missing),0,
7,4 Consistency,Dates written with '/' converted to '-',5,
8,4 Validity,Unparseable dates,0,
9,5 Validity,Non-numeric amount text set to missing,0,


Row count before: 612 | after: 600 | removed: 12


,0
txn_id,string[python]
agent_id,string[python]
txn_type,object
amount,float64
txn_date,datetime64[ns]
is_outlier,bool
amount_imputed,bool
county,object
onboarded,datetime64[ns]
unknown_agent,bool


## 3. Decisions
- **Duplicates:** [N] repeated `txn_id` rows removed (keep first). A transaction ID can occur only once, so repeats double-count money.
- **Missing/invalid amounts:** [N] rows (empty + "N/A"). I imputed the median amount for the same `txn_type` and set `amount_imputed=True`.
  Dropping them would understate counts; the median is robust to extreme values. Imputed values are estimates, so the flag lets anyone exclude them.
- **Unknown agents:** [N] rows reference AG099, which is not in the register. Kept with county "Unknown" (left join) and flagged in `unknown_agent`, because the money still moved.

In [5]:
big = clean[clean["amount"] == 250000]
display(big[["txn_id", "agent_id", "txn_type", "txn_date", "county", "amount"]])

normal = clean.loc[(clean["amount"] != 250000) & ~clean["amount_imputed"], "amount"]
print("Median:", normal.median(), "| 99th pct:", normal.quantile(.99), "| Max (excl. 250k):", normal.max())
print("Rows flagged is_outlier:", clean["is_outlier"].sum())
print("Share of total value from the 250k rows:", round(big["amount"].sum()/clean["amount"].sum()*100, 1), "%")

,txn_id,agent_id,txn_type,txn_date,county,amount
306,TX00307,AG012,Deposit,2026-03-02,Kisumu,250000.0
315,TX00316,AG014,Deposit,2026-04-01,Nairobi,250000.0
424,TX00425,AG005,Withdrawal,2026-03-02,Mombasa,250000.0
463,TX00464,AG009,Withdrawal,2026-04-11,Nakuru,250000.0
594,TX00595,AG016,Deposit,2026-03-11,Kiambu,250000.0


Median: 1770.0 | 99th pct: 9609.059999999998 | Max (excl. 250k): 28847.0
Rows flagged is_outlier: 40
Share of total value from the 250k rows: 46.5 %


### Outlier recommendation
There are [N] rows with an identical round amount of 250,000, across different agents and dates. The largest normal amount is about [X],
so these are roughly [Y]× larger. Identical repeated values suggest a **system default or cap**, not genuine customer behaviour.
**Recommendation:** do not delete them; keep them flagged, exclude them from average-value reporting until the source-system owner confirms them,
and fix the root cause. Other IQR outliers are expected in right-skewed transaction data and are retained.

In [6]:
expected = txns_raw["txn_id"].nunique()
print("Raw rows:", len(txns_raw), "| Unique txn_id in raw:", expected, "| Rows after merge:", len(clean))
assert len(clean) == expected
print("Transactions with unknown agents:", clean["unknown_agent"].sum())
print(clean.loc[clean["unknown_agent"], "agent_id"].value_counts())

Raw rows: 612 | Unique txn_id in raw: 600 | Rows after merge: 600
Transactions with unknown agents: 7
agent_id
AG099    7
Name: count, dtype: Int64


## 4. Summary tables

In [7]:
county_summary = (clean.groupby(["county", "txn_type"])
                  .agg(total_value=("amount", "sum"),
                       n_transactions=("txn_id", "count"),
                       avg_value=("amount", "mean"))
                  .round(0).reset_index())
display(county_summary)

clean["week_start"] = clean["txn_date"].dt.to_period("W").dt.start_time.dt.date
weekly_pivot = pd.pivot_table(clean, index="week_start", columns="county",
                              values="amount", aggfunc="sum", fill_value=0, margins=True).round(0)
weekly_pivot

,county,txn_type,total_value,n_transactions,avg_value
0,Kiambu,Deposit,334737.0,43,7785.0
1,Kiambu,Withdrawal,77024.0,27,2853.0
2,Kisumu,Deposit,436502.0,74,5899.0
3,Kisumu,Withdrawal,76348.0,37,2063.0
4,Mombasa,Deposit,149094.0,72,2071.0
5,Mombasa,Withdrawal,368592.0,61,6042.0
6,Nairobi,Deposit,543698.0,118,4608.0
7,Nairobi,Withdrawal,161932.0,62,2612.0
8,Nakuru,Deposit,184944.0,60,3082.0
9,Nakuru,Withdrawal,328568.0,39,8425.0


county,Kiambu,Kisumu,Mombasa,Nairobi,Nakuru,Unknown,All
week_start,,,,,,,
2026-02-23,0.0,1893.0,6759.0,7046.0,2770.0,0.0,18468.0
2026-03-02,18317.0,287933.0,285696.0,68967.0,56574.0,0.0,717488.0
2026-03-09,267746.0,23186.0,30504.0,52888.0,28730.0,0.0,403054.0
2026-03-16,22054.0,28126.0,25172.0,43134.0,39795.0,1639.0,159920.0
2026-03-23,11504.0,27077.0,17050.0,41400.0,18972.0,0.0,116004.0
2026-03-30,22609.0,41077.0,38344.0,282931.0,18145.0,8507.0,411613.0
2026-04-06,15030.0,15341.0,47120.0,56814.0,276609.0,9824.0,420737.0
2026-04-13,12265.0,29470.0,26730.0,56406.0,26180.0,0.0,151051.0
2026-04-20,24507.0,37439.0,16687.0,65135.0,30862.0,1877.0,176507.0


In [8]:
clean.drop(columns=["week_start"]).to_csv("outputs/agent_transactions_clean.csv", index=False)

chk = pd.read_csv("outputs/agent_transactions_clean.csv")
assert chk["txn_id"].is_unique
assert set(chk["txn_type"]) == {"Deposit", "Withdrawal"}
assert {"amount_imputed", "is_outlier", "county"} <= set(chk.columns)
print(chk.shape, chk["txn_date"].head(3).tolist())

(600, 10) ['2026-03-03', '2026-04-27', '2026-04-29']


# Data Quality Report
**Scope:** 612 raw transactions (20 agents, 5 counties) → 600 clean rows. [adjust to your output]

| Dimension | Issue found | Rows | Treatment |
|---|---|---|---|
| Completeness | Missing amounts (empty + "N/A") | [ ] | Median imputation by type; `amount_imputed` flag |
| Uniqueness | Repeated txn_id | [ ] | Removed, keeping first |
| Validity | Unknown agent_id (AG099) | [ ] | Kept, county "Unknown", `unknown_agent` flag |
| Consistency | Mixed txn_type labels; mixed date formats | [ ] | Standardised to Deposit/Withdrawal; ISO dates |
| Accuracy | 250,000 values; IQR outliers | [ ] | Flagged, not deleted; verify with source |
| Timeliness | Dates outside the expected 60-day window or in the future | 0 | No action needed |

**Limitations:** imputed values are estimates; the 250,000 rows are unresolved; [add your own].

**Privacy (Kenya Data Protection Act 2019):** If the extract also held customer phone numbers, the phone number (MSISDN) would be
directly identifying personal data. Combined with amounts, dates and agent locations it would reveal individuals' financial behaviour,
so the whole record would count as personal data, not only the phone field. Following the Act's principles of data minimisation and
purpose limitation, I would exclude phone numbers from the analysis extract; where a customer link is needed, I would replace them with a
salted-hash pseudonym, with the key stored separately, restrict access by role, encrypt data at rest and in transit, retain it only as long
as needed, and ensure the controller is registered with the Office of the Data Protection Commissioner.